
# 24B — V5 Astrology Discovery Tournament — Automatic Feature Selection

This **supersedes 24A before any V5 astrology feature-generation run**.

The methodology is intentionally simpler:

- Freeze **one broad, theory-bounded astrology feature universe**.
- Use only **two winner-eligible models** over exactly the same universe:
  1. Ridge: keep everything but shrink automatically.
  2. ElasticNet: automatically select/shrink features **inside inner subject-CV**.
- Keep TG10 as a diagnostic sanity check only.
- Use nuisance-only and nuisance+astrology only to measure dataset bias and incremental astrology signal.

There is **no manual post-score feature picking**.

### Hard exclusions
- No CONFIRM.
- No Production Control.
- No axis-specific astrology coefficients.
- No event/pair changes.
- No outcome-based univariate feature screening.
- No recursively flattened shinsal runtime tree in the primary universe.


In [5]:

from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
import ast, hashlib, json, math, re, sys, warnings

import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

NOTEBOOK_VERSION = "SAJU_ML_V5_ASTROLOGY_DISCOVERY_TOURNAMENT_AUTOSELECT_20260817"
SEED = 20260817
OUTER_REPEATS = 5
OUTER_FOLDS = 5
INNER_FOLDS = 3
N_BOOTSTRAP = 10000

RIDGE_C_GRID = [0.03, 0.10, 0.30, 1.00, 3.00]
ELASTIC_C_GRID = [0.03, 0.10, 0.30, 1.00]
ELASTIC_L1_GRID = [0.25, 0.50, 0.75, 1.00]

MIN_ASTRO_MACRO = 0.55
MIN_ASTRO_P10 = 0.48
MIN_CONDITIONAL_DELTA = 0.01
MIN_BOOT_P_DELTA = 0.80
MIN_BROAD_ASTRO = 0.52
MIN_BROAD_DELTA = 0.00
MIN_SUPPORTED_WAVE = 0.45
SUPPORTED_WAVE_MIN_SUBJECTS = 10

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for c in [p] + list(p.parents):
        if (c / "saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside the Chartpalja repository (saju_engine.py required).")

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

ROOT = find_repo_root()

FAST = ROOT / "research/ml/artifacts/v5_discovery_fasttrack"
PRIMARY_PAIRS = FAST / "V5_FASTTRACK_COMBINED_LOCAL_PAIRS_PRIMARY_TRUSTED.csv"
BROAD_PAIRS = FAST / "V5_FASTTRACK_COMBINED_LOCAL_PAIRS_BROAD_SENSITIVITY.csv"
READY = FAST / "V5_FASTTRACK_DEV_READINESS_DECISION.json"
NUISANCE_23A = FAST / "V5_FASTTRACK_NUISANCE_DIAGNOSTICS.json"

ORIG_ROSTER = ROOT / "research/ml/artifacts/v5_identity_repair/V5_DEV_SUBJECT_ROSTER_160_REPAIRED.csv"
E1_ROSTER = ROOT / "research/ml/artifacts/v5_dev_expansion_e1/V5_DEV_EXPANSION_E1_ROSTER_160.csv"
E2_ROSTER = ROOT / "research/ml/artifacts/v5_dev_expansion_e2/V5_DEV_EXPANSION_E2_ROSTER_160.csv"

BIRTH_SNAPSHOT = ROOT / "research/ml/artifacts/v4_unified_dev_roster/PersonList-15k.csv"
PREREG = ROOT / "research/ml_corpus/v5_ground_truth/V5_DISCOVERY_ASTROLOGY_TOURNAMENT_PREREGISTRATION_V2.json"

ART = ROOT / "research/ml/artifacts/v5_astrology_discovery_tournament"
ART.mkdir(parents=True, exist_ok=True)

for p in [
    PRIMARY_PAIRS, BROAD_PAIRS, READY, NUISANCE_23A,
    ORIG_ROSTER, E1_ROSTER, E2_ROSTER, BIRTH_SNAPSHOT, PREREG
]:
    if not p.exists():
        raise FileNotFoundError(p)

print("ROOT:", ROOT)
print("OUTPUT:", ART)


ROOT: /Users/sangjinlee/Desktop/projects/saju
OUTPUT: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v5_astrology_discovery_tournament


## 1. Frozen-pair and no-leakage preflight

In [6]:

ready = json.load(open(READY, encoding="utf-8"))
protocol = json.load(open(PREREG, encoding="utf-8"))
primary = pd.read_csv(PRIMARY_PAIRS)
broad = pd.read_csv(BROAD_PAIRS)

assert ready["status"] == "V5_DISCOVERY_FASTTRACK_READY_FOR_ASTROLOGY_AND_MODEL_DEVELOPMENT"
assert ready["astrology_generation_allowed"] is True
assert ready["control_scoring_allowed"] is False
assert ready["confirm_event_research_allowed"] is False
assert ready["rules"]["confirm_remains_sealed"] is True
assert protocol["status"] == "PREDECLARED_BEFORE_V5_ASTROLOGY_FEATURE_GENERATION_REVISED_AUTO_SELECTION"

assert sha256_file(PRIMARY_PAIRS) == ready["combined_primary"]["pairs_sha256"]
assert sha256_file(BROAD_PAIRS) == ready["combined_broad"]["pairs_sha256"]
assert len(primary) == ready["combined_primary"]["pair_rows_n"]
assert len(broad) == ready["combined_broad"]["pair_rows_n"]
assert primary.subject_id.nunique() == ready["combined_primary"]["pairable_subjects"]["TOTAL"]
assert broad.subject_id.nunique() == ready["combined_broad"]["pairable_subjects"]["TOTAL"]

required_pair_cols = {
    "subject_id","name","preassigned_axis","collection_wave","birth_year",
    "earlier_year","later_year","year_gap","earlier_is_positive","subject_weight"
}
for label, frame in [("PRIMARY", primary), ("BROAD", broad)]:
    missing = required_pair_cols - set(frame.columns)
    assert not missing, f"{label} missing columns: {sorted(missing)}"
    assert frame.earlier_is_positive.isin([0,1]).all()
    assert frame.year_gap.between(1,5).all()
    assert (frame.earlier_year < frame.later_year).all()

# Explicitly do not even define a CONFIRM path in this notebook.
CONFIRM_LOADED = False
CONTROL_SCORED = False

print("23A lineage: PASS")
print("PRIMARY:", len(primary), "pairs /", primary.subject_id.nunique(), "subjects")
print("BROAD  :", len(broad), "pairs /", broad.subject_id.nunique(), "subjects")
print("CONFIRM loaded:", CONFIRM_LOADED)
print("Control scored:", CONTROL_SCORED)


23A lineage: PASS
PRIMARY: 193 pairs / 100 subjects
BROAD  : 197 pairs / 104 subjects
CONFIRM loaded: False
Control scored: False



## 2. Rebuild exact engine inputs from frozen rosters

The event worklists intentionally omitted birth time.  
The full frozen rosters retain birth lineage; if a roster field is absent, this notebook may fill it **only** by `source_row_key` from the already frozen `PersonList-15k.csv`.

It never guesses a birth hour.


In [7]:

orig_r = pd.read_csv(ORIG_ROSTER)
e1_r = pd.read_csv(E1_ROSTER)
e2_r = pd.read_csv(E2_ROSTER)

orig_r["collection_wave"] = "ORIGINAL"
e1_r["collection_wave"] = "E1"
e2_r["collection_wave"] = "E2"

roster = pd.concat([orig_r, e1_r, e2_r], ignore_index=True, sort=False)
assert roster.subject_id.nunique() == 480
assert roster.subject_id.duplicated().sum() == 0

needed_subjects = set(broad.subject_id.astype(str))
r = roster[roster.subject_id.astype(str).isin(needed_subjects)].copy()
assert set(r.subject_id.astype(str)) == needed_subjects

snapshot = pd.read_csv(BIRTH_SNAPSHOT)
required_snapshot_cols = {"RowKey","BirthTime","Gender","Name","Notes"}
assert required_snapshot_cols.issubset(snapshot.columns)

def parse_birth_blob(blob):
    obj = json.loads(str(blob))
    std = obj["StdTime"]
    loc = obj["Location"]
    m = re.match(
        r"^(\d{1,2}):(\d{2})\s+(\d{2})/(\d{2})/(\d{4})\s+([+-]\d{2}:\d{2})$",
        std.strip()
    )
    if not m:
        raise ValueError(std)
    hh, mm, dd, mo, yyyy, offset = m.groups()
    return {
        "snapshot_birth_date": "%04d-%02d-%02d" % (int(yyyy), int(mo), int(dd)),
        "snapshot_birth_time": "%02d:%02d" % (int(hh), int(mm)),
        "snapshot_utc_offset": offset,
        "snapshot_birth_place": loc["Name"],
        "snapshot_longitude": float(loc["Longitude"]),
        "snapshot_latitude": float(loc["Latitude"]),
    }

snap_rows = []
for _, row in snapshot.iterrows():
    try:
        parsed = parse_birth_blob(row["BirthTime"])
    except Exception:
        continue
    snap_rows.append({
        "source_row_key": str(row["RowKey"]),
        **parsed
    })
snap = pd.DataFrame(snap_rows).drop_duplicates("source_row_key")

r["source_row_key"] = r["source_row_key"].astype(str)
r = r.merge(snap, on="source_row_key", how="left", validate="many_to_one")

for target, source in [
    ("birth_date","snapshot_birth_date"),
    ("birth_time","snapshot_birth_time"),
    ("utc_offset","snapshot_utc_offset"),
    ("birth_place","snapshot_birth_place"),
    ("longitude","snapshot_longitude"),
    ("latitude","snapshot_latitude"),
]:
    if target not in r.columns:
        r[target] = np.nan
    missing = r[target].isna() | (r[target].astype(str).str.strip() == "")
    r.loc[missing, target] = r.loc[missing, source]

# Frozen date and snapshot date must agree for every paired subject.
assert r["snapshot_birth_date"].notna().all(), "A paired subject does not resolve to frozen PersonList-15k by source_row_key."
assert (
    pd.to_datetime(r.birth_date).dt.strftime("%Y-%m-%d")
    == pd.to_datetime(r.snapshot_birth_date).dt.strftime("%Y-%m-%d")
).all(), "Roster birth date disagrees with frozen birth snapshot."

for c in ["birth_date","birth_time","utc_offset","longitude","latitude","gender"]:
    assert r[c].notna().all(), f"Missing exact engine input: {c}"
    assert ~(r[c].astype(str).str.strip() == "").any(), f"Blank exact engine input: {c}"

# Pair metadata axis/wave must agree with roster lineage.
pair_meta = broad[["subject_id","preassigned_axis","collection_wave"]].drop_duplicates()
chk = pair_meta.merge(
    r[["subject_id","preassigned_axis","collection_wave"]],
    on="subject_id", how="left", suffixes=("_pair","_roster"), validate="one_to_one"
)
assert (chk.preassigned_axis_pair == chk.preassigned_axis_roster).all()
assert (chk.collection_wave_pair == chk.collection_wave_roster).all()

engine_input_cols = [
    "subject_id","name","preassigned_axis","collection_wave","gender",
    "birth_date","birth_time","utc_offset","birth_place","latitude","longitude",
    "source_row_key","rodden_rating"
]
engine_inputs = r[[c for c in engine_input_cols if c in r.columns]].copy()
engine_inputs.to_csv(ART / "V5_DISCOVERY_ENGINE_INPUT_AUDIT.csv", index=False)

print("Exact-time engine inputs:", len(engine_inputs))
display(engine_inputs.groupby("collection_wave").size().rename("paired_subjects"))


Exact-time engine inputs: 104


collection_wave
E1          29
E2          22
ORIGINAL    53
Name: paired_subjects, dtype: int64

## 3. Canonical engine helpers and exact Ten-God mapping

In [8]:
sys.path.insert(0, str(ROOT))

import saju_engine as se
import sajupy

STEMS = list("甲乙丙丁戊己庚辛壬癸")
BRANCHES = list("子丑寅卯辰巳午未申酉戌亥")
ELEMENTS = ["wood", "fire", "earth", "metal", "water"]

STEM_ELEMENT_EN = {
    "甲":"wood","乙":"wood","丙":"fire","丁":"fire","戊":"earth",
    "己":"earth","庚":"metal","辛":"metal","壬":"water","癸":"water"
}
STEM_YINYANG = {
    "甲":"yang","乙":"yin","丙":"yang","丁":"yin","戊":"yang",
    "己":"yin","庚":"yang","辛":"yin","壬":"yang","癸":"yin"
}
BRANCH_ELEMENT_EN = {
    "子":"water","丑":"earth","寅":"wood","卯":"wood","辰":"earth","巳":"fire",
    "午":"fire","未":"earth","申":"metal","酉":"metal","戌":"earth","亥":"water"
}
HIDDEN_STEMS = {
    "子":["癸"], "丑":["己","癸","辛"], "寅":["甲","丙","戊"], "卯":["乙"],
    "辰":["戊","乙","癸"], "巳":["丙","戊","庚"], "午":["丁","己"],
    "未":["己","丁","乙"], "申":["庚","壬","戊"], "酉":["辛"],
    "戌":["戊","辛","丁"], "亥":["壬","甲"]
}

GENERATES = {
    "wood":"fire","fire":"earth","earth":"metal","metal":"water","water":"wood"
}
CONTROLS = {
    "wood":"earth","earth":"water","water":"fire","fire":"metal","metal":"wood"
}

STEM_COMBINES = {
    frozenset(x)
    for x in [("甲","己"),("乙","庚"),("丙","辛"),("丁","壬"),("戊","癸")]
}
STEM_CLASHES = {
    frozenset(x)
    for x in [("甲","庚"),("乙","辛"),("丙","壬"),("丁","癸")]
}
BRANCH_COMBINES = {
    frozenset(x)
    for x in [("子","丑"),("寅","亥"),("卯","戌"),("辰","酉"),("巳","申"),("午","未")]
}
BRANCH_CLASHES = {
    frozenset(x)
    for x in [("子","午"),("丑","未"),("寅","申"),("卯","酉"),("辰","戌"),("巳","亥")]
}
BRANCH_HARMS = {
    frozenset(x)
    for x in [("子","未"),("丑","午"),("寅","巳"),("卯","辰"),("申","亥"),("酉","戌")]
}
BRANCH_BREAKS = {
    frozenset(x)
    for x in [("子","酉"),("丑","辰"),("寅","亥"),("卯","午"),("巳","申"),("未","戌")]
}
SELF_PUNISH = set(["辰","午","酉","亥"])
PUNISH_TRIPLES = [set("寅巳申"), set("丑未戌")]

UNSEONG_STATES = [
    "장생","목욕","관대","건록","제왕","쇠",
    "병","사","묘","절","태","양"
]
UNSEONG_START = {
    "甲":"亥", "乙":"午",
    "丙":"寅", "丁":"酉",
    "戊":"寅", "己":"酉",
    "庚":"巳", "辛":"子",
    "壬":"申", "癸":"卯",
}
YANG_STEMS = set("甲丙戊庚壬")
SHINSAL_KEYWORDS = ("신살", "귀인", "공망")

def parse_utc_offset(raw):
    if isinstance(raw, (int, float)):
        return float(raw)
    s = str(raw).strip()
    sign = -1.0 if s.startswith("-") else 1.0
    s = s[1:] if s[:1] in "+-" else s
    hh, mm = s.split(":")
    return sign * (int(hh) + int(mm) / 60.0)

def public_lon_corrected_birth(public_birth):
    y, m, d = map(int, public_birth["date"].split("-"))
    hh, mi = map(int, public_birth["time"].split(":")[:2])
    lon = float(public_birth["longitude"])
    utc_hours = parse_utc_offset(public_birth["utc_offset"])

    calc = sajupy.get_saju_calculator()
    civil = datetime(y, m, d, hh, mi)

    corr = float(
        calc._calculate_solar_time_correction(lon, utc_hours)
    )
    h2, min2, dc = calc._adjust_time_for_solar(
        civil.hour, civil.minute, corr
    )
    y2, m2, d2 = calc._adjust_date_for_solar(
        civil.year, civil.month, civil.day, dc
    )

    return {
        "calendar": public_birth.get("calendar", "solar"),
        "y": y2, "m": m2, "d": d2,
        "h": h2, "min": min2,
    }

def pillar_chars(value):
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        a, b = str(value[0]), str(value[1])
        if a in STEMS and b in BRANCHES:
            return a, b

    s = str(value)
    stem = next((c for c in s if c in STEMS), None)
    branch = next((c for c in s if c in BRANCHES), None)
    return stem, branch

def sexagenary_year_pillar(year):
    i = (int(year) - 1984) % 60
    return STEMS[i % 10] + BRANCHES[i % 12]

def ten_god_group(day_stem, other_stem):
    de = STEM_ELEMENT_EN[day_stem]
    oe = STEM_ELEMENT_EN[other_stem]

    if de == oe:
        return "peer"
    if GENERATES[de] == oe:
        return "output"
    if CONTROLS[de] == oe:
        return "wealth"
    if CONTROLS[oe] == de:
        return "officer"
    if GENERATES[oe] == de:
        return "resource"

    raise RuntimeError((day_stem, other_stem))

def onehot(out, prefix, value, vocab):
    for v in vocab:
        out[prefix + str(v)] = float(value == v)

def pair_relation(a, b, relation_set):
    if a is None or b is None:
        return 0.0
    return float(frozenset((a, b)) in relation_set)

def branch_punishment(a, b):
    if a is None or b is None:
        return 0.0
    if a == b and a in SELF_PUNISH:
        return 1.0
    if set([a, b]) == set(["子", "卯"]):
        return 1.0
    return float(any(set([a, b]).issubset(x) for x in PUNISH_TRIPLES))

def get_strength_label(result):
    x = result.get("신강신약")
    if isinstance(x, dict):
        return x.get("판정") or x.get("label")
    if isinstance(x, str):
        return x
    return None

def get_daewoon_pillar(meta_row):
    for key in ("대운_pillar", "대운", "daewoon_pillar"):
        if meta_row.get(key):
            st, br = pillar_chars(meta_row[key])
            if st and br:
                return st + br
    return None

def get_daewoon_row(daewoon, year):
    for row in daewoon:
        if int(row["start_year"]) <= int(year) < int(row["end_year"]):
            return row
    return None

def get_sewoon_pillar(meta_row, year):
    for key in ("세운_pillar", "세운", "연주", "year_pillar"):
        if meta_row.get(key):
            st, br = pillar_chars(meta_row[key])
            if st and br:
                return st + br
    return sexagenary_year_pillar(year)

def control_score_from_meta(meta_row):
    candle = meta_row.get("candle") or {}
    value = candle.get("close")
    return float(value) if value is not None else np.nan

def local_twelve_unseong(day_stem, branch):
    start = UNSEONG_START[day_stem]
    start_i = BRANCHES.index(start)
    branch_i = BRANCHES.index(branch)
    direction = 1 if day_stem in YANG_STEMS else -1
    steps = ((branch_i - start_i) * direction) % 12
    return UNSEONG_STATES[steps]

def get_unseong_state(day_stem, branch):
    for fn_name in ("twelve_unseong", "_twelve_unseong"):
        fn = getattr(se, fn_name, None)
        if fn is None:
            continue

        for args in ((day_stem, branch), (branch, day_stem)):
            try:
                value = fn(*args)
                if isinstance(value, str) and value in UNSEONG_STATES:
                    return value
            except Exception:
                pass

    return local_twelve_unseong(day_stem, branch)

def strength_bucket(result):
    x = result.get("신강신약") or {}
    verdict = x.get("판정") if isinstance(x, dict) else str(x)
    verdict = verdict or ""

    if any(k in verdict for k in ("신강", "태강", "극왕")):
        return "strong"
    if any(k in verdict for k in ("신약", "태약", "극약")):
        return "weak"
    return "neutral"

# V1.3 Yongshin used the engine's element labels.
STEM_ELEMENT_ENGINE = getattr(se, "STEM_ELEMENT", {
    "甲":"목","乙":"목","丙":"화","丁":"화","戊":"토",
    "己":"토","庚":"금","辛":"금","壬":"수","癸":"수"
})
BRANCH_ELEMENT_ENGINE = getattr(se, "BRANCH_ELEMENT_MAIN", {
    "子":"수","丑":"토","寅":"목","卯":"목","辰":"토","巳":"화",
    "午":"화","未":"토","申":"금","酉":"금","戌":"토","亥":"수"
})

def yongshin_block_features(result, stem, branch, prefix):
    out = {}
    yong = result.get("용신") or {}
    day_stem = (result.get("원국") or {}).get("day", ["", ""])[0]

    fit = {}

    if yong and day_stem and hasattr(se, "_check_yongshin_fit"):
        try:
            fit = se._check_yongshin_fit(
                stem, branch, yong, day_stem
            ) or {}
        except Exception:
            fit = {}

    mapping = [
        ("용신부합", "yong_fit"),
        ("희신부합", "hee_fit"),
        ("기신부합", "gi_fit"),
        ("구신부합", "gu_fit"),
        ("용신부합_천간", "yong_stem_fit"),
        ("용신부합_지지", "yong_branch_fit"),
    ]

    for source_key, target_key in mapping:
        out[
            "yongshin__%s_%s" % (prefix, target_key)
        ] = float(fit.get(source_key) or 0.0)

    yong_e = yong.get("용신_오행") or ""
    hee = set(yong.get("희신_오행") or [])
    gi = set(yong.get("기신_오행") or [])
    gu = set(yong.get("구신_오행") or [])

    fav = (set([yong_e]) | hee) - set([""])
    unfav = (gi | gu) - set([""])

    stem_e = STEM_ELEMENT_ENGINE.get(stem, "")
    branch_e = BRANCH_ELEMENT_ENGINE.get(branch, "")

    out["yongshin__%s_supplies_fav" % prefix] = float(
        stem_e in fav or branch_e in fav
    )
    out["yongshin__%s_supplies_unfav" % prefix] = float(
        stem_e in unfav or branch_e in unfav
    )

    return out

def unseong_block_features(result, branch, prefix):
    out = {}
    day_stem = (result.get("원국") or {}).get("day", ["", ""])[0]
    state = get_unseong_state(day_stem, branch)

    for state_name in UNSEONG_STATES:
        out[
            "unseong__%s_state_%s" % (prefix, state_name)
        ] = float(state == state_name)

    raw_map = getattr(se, "_UNSEONG_SCORE", {})
    if isinstance(raw_map, dict):
        raw_score = float(raw_map.get(state, 0.0))
    else:
        raw_score = 0.0

    out["unseong__%s_raw_score" % prefix] = raw_score

    bucket = strength_bucket(result)
    sign = 1.0 if bucket == "weak" else (
        -1.0 if bucket == "strong" else 0.0
    )

    out["unseong__%s_score_x_strength" % prefix] = (
        raw_score * sign
    )

    return out



In [9]:
EXACT_TENGODS = [
    "bijian", "jiecai", "shishen", "shangguan",
    "pian_cai", "zheng_cai", "qi_sha", "zheng_guan",
    "pian_yin", "zheng_yin",
]

def exact_ten_god(day_stem, other_stem):
    de = STEM_ELEMENT_EN[day_stem]
    oe = STEM_ELEMENT_EN[other_stem]
    same_polarity = STEM_YINYANG[day_stem] == STEM_YINYANG[other_stem]

    if de == oe:
        return "bijian" if same_polarity else "jiecai"
    if GENERATES[de] == oe:
        return "shishen" if same_polarity else "shangguan"
    if CONTROLS[de] == oe:
        return "pian_cai" if same_polarity else "zheng_cai"
    if CONTROLS[oe] == de:
        return "qi_sha" if same_polarity else "zheng_guan"
    if GENERATES[oe] == de:
        return "pian_yin" if same_polarity else "zheng_yin"
    raise RuntimeError((day_stem, other_stem))

def exact_tg_distribution(day_stem, stems):
    counts = Counter(exact_ten_god(day_stem, st) for st in stems)
    n = float(max(1, len(stems)))
    return {tg: counts[tg] / n for tg in EXACT_TENGODS}

## 4. V5 shared-coefficient astrology primitive extractor

In [10]:

def subject_public_birth(subject):
    return {
        "calendar": "solar",
        "date": str(subject["birth_date"]),
        "time": str(subject["birth_time"]),
        "utc_offset": str(subject["utc_offset"]),
        "longitude": float(subject["longitude"]),
    }

ENGINE_CALCULATOR = sajupy.get_saju_calculator()
ENGINE_MIN_YEAR = int(ENGINE_CALCULATOR.min_year)
ENGINE_MAX_YEAR = int(ENGINE_CALCULATOR.max_year)

def compute_subject_engine(subject):
    corrected = public_lon_corrected_birth(subject_public_birth(subject))
    if not (ENGINE_MIN_YEAR <= int(corrected["y"]) <= ENGINE_MAX_YEAR):
        raise ValueError((subject["subject_id"], corrected["y"], ENGINE_MIN_YEAR, ENGINE_MAX_YEAR))

    inp = se.BirthInput(
        year=int(corrected["y"]),
        month=int(corrected["m"]),
        day=int(corrected["d"]),
        hour=int(corrected["h"]),
        minute=int(corrected["min"]),
        gender=str(subject["gender"]),
        calendar="solar",
        is_leap_month=False,
        use_solar_time=False,
        utc_offset=9,
    )

    result = se.compute_all(inp)
    daewoon = se.build_daewoon_detail(result)
    timeline = result["chart_data"]["연도별_타임라인"]
    meta = {int(row["year"]): row for row in timeline}
    return result, daewoon, meta

def extract_v5_primitives(subject, year, result, daewoon, meta_row):
    natal = result["원국"]
    natal_pillars = {
        pos: pillar_chars(natal[pos])
        for pos in ("year","month","day","hour")
    }

    day_stem = natal_pillars["day"][0]
    dw_stem, dw_branch = pillar_chars(get_daewoon_pillar(meta_row))
    sw_stem, sw_branch = pillar_chars(get_sewoon_pillar(meta_row, year))

    if not all([day_stem,dw_stem,dw_branch,sw_stem,sw_branch]):
        raise ValueError(f"Pillar parse failed: {subject['subject_id']} {year}")

    f = {}

    # Dynamic basic states only (shared across all axes).
    for label, stem, branch in [("dw",dw_stem,dw_branch),("sw",sw_stem,sw_branch)]:
        onehot(f, f"basic__{label}_element_", STEM_ELEMENT_EN[stem], ELEMENTS)
        onehot(f, f"basic__{label}_yy_", STEM_YINYANG[stem], ["yang","yin"])
        onehot(f, f"basic__{label}_branch_", branch, BRANCHES)

    # Exact 10-Ten-God stem + branch hidden-stem distribution.
    strength = strength_bucket(result)
    for label, stem, branch in [("dw",dw_stem,dw_branch),("sw",sw_stem,sw_branch)]:
        tg = exact_ten_god(day_stem, stem)
        onehot(f, f"tg10__{label}_stem_", tg, EXACT_TENGODS)

        dist = exact_tg_distribution(day_stem, HIDDEN_STEMS[branch])
        for exact_name in EXACT_TENGODS:
            f[f"tg10__{label}_branch_hidden_{exact_name}"] = float(dist[exact_name])

        dynamic = {
            k: 0.5 * (float(tg == k) + float(dist[k]))
            for k in EXACT_TENGODS
        }
        for exact_name in EXACT_TENGODS:
            for bucket in ["weak","neutral","strong"]:
                f[f"tgxstrength__{label}_{exact_name}_{bucket}"] = (
                    dynamic[exact_name] * float(strength == bucket)
                )

    # Dynamic-to-natal and Daewoon↔Sewoon relations.
    natal_stems = [x[0] for x in natal_pillars.values()]
    natal_branches = [x[1] for x in natal_pillars.values()]

    for label, stem, branch in [("dw",dw_stem,dw_branch),("sw",sw_stem,sw_branch)]:
        f[f"relation__{label}_stem_combine_natal"] = sum(
            pair_relation(stem, x, STEM_COMBINES) for x in natal_stems
        )
        f[f"relation__{label}_stem_clash_natal"] = sum(
            pair_relation(stem, x, STEM_CLASHES) for x in natal_stems
        )
        f[f"relation__{label}_branch_combine_natal"] = sum(
            pair_relation(branch, x, BRANCH_COMBINES) for x in natal_branches
        )
        f[f"relation__{label}_branch_clash_natal"] = sum(
            pair_relation(branch, x, BRANCH_CLASHES) for x in natal_branches
        )
        f[f"relation__{label}_branch_harm_natal"] = sum(
            pair_relation(branch, x, BRANCH_HARMS) for x in natal_branches
        )
        f[f"relation__{label}_branch_break_natal"] = sum(
            pair_relation(branch, x, BRANCH_BREAKS) for x in natal_branches
        )
        f[f"relation__{label}_branch_punish_natal"] = sum(
            branch_punishment(branch, x) for x in natal_branches
        )

    f["relation__sw_dw_stem_combine"] = pair_relation(sw_stem,dw_stem,STEM_COMBINES)
    f["relation__sw_dw_stem_clash"] = pair_relation(sw_stem,dw_stem,STEM_CLASHES)
    f["relation__sw_dw_branch_combine"] = pair_relation(sw_branch,dw_branch,BRANCH_COMBINES)
    f["relation__sw_dw_branch_clash"] = pair_relation(sw_branch,dw_branch,BRANCH_CLASHES)
    f["relation__sw_dw_branch_harm"] = pair_relation(sw_branch,dw_branch,BRANCH_HARMS)
    f["relation__sw_dw_branch_break"] = pair_relation(sw_branch,dw_branch,BRANCH_BREAKS)
    f["relation__sw_dw_branch_punish"] = branch_punishment(sw_branch,dw_branch)

    # Orthodox engine-context primitives.
    f.update(yongshin_block_features(result,dw_stem,dw_branch,"dw"))
    f.update(yongshin_block_features(result,sw_stem,sw_branch,"sw"))
    f.update(unseong_block_features(result,dw_branch,"dw"))
    f.update(unseong_block_features(result,sw_branch,"sw"))

    # Deliberately NO candle.close / Control score and NO flattened shinsal.
    return f


## 5. Generate one frozen year-feature cache for PRIMARY ∪ BROAD

In [11]:

def add_positive_negative_years(frame):
    x = frame.copy()
    x["positive_year"] = np.where(
        x.earlier_is_positive.astype(int).eq(1),
        x.earlier_year, x.later_year
    ).astype(int)
    x["negative_year"] = np.where(
        x.earlier_is_positive.astype(int).eq(1),
        x.later_year, x.earlier_year
    ).astype(int)
    x["positive_earlier_calc"] = (x.positive_year < x.negative_year).astype(int)
    x["positive_age"] = x.positive_year - x.birth_year.astype(int)
    x["negative_age"] = x.negative_year - x.birth_year.astype(int)
    x["abs_year_gap"] = (x.positive_year - x.negative_year).abs().astype(int)
    return x

primary = add_positive_negative_years(primary)
broad = add_positive_negative_years(broad)

required_years = defaultdict(set)
for frame in [primary,broad]:
    for _, row in frame.iterrows():
        required_years[str(row.subject_id)].add(int(row.positive_year))
        required_years[str(row.subject_id)].add(int(row.negative_year))

subject_map = {
    str(row.subject_id): row.to_dict()
    for _, row in engine_inputs.iterrows()
}
assert set(required_years).issubset(subject_map)

rows = []
engine_failures = []

for i, sid in enumerate(sorted(required_years)):
    subject = subject_map[sid]
    try:
        result, daewoon, meta = compute_subject_engine(subject)
        for year in sorted(required_years[sid]):
            if year not in meta:
                raise RuntimeError(f"Timeline missing {sid} {year}")
            feat = extract_v5_primitives(subject,year,result,daewoon,meta[year])
            rows.append({"subject_id":sid,"year":int(year),**feat})
    except Exception as e:
        engine_failures.append({
            "subject_id":sid,
            "name":subject.get("name"),
            "error":repr(e)
        })

    if (i+1) % 10 == 0:
        print("engine subjects:", i+1, "/", len(required_years))

failure_path = ART / "V5_DISCOVERY_ENGINE_FEATURE_FAILURES.csv"
pd.DataFrame(engine_failures).to_csv(failure_path,index=False)

if engine_failures:
    raise RuntimeError(
        f"Astrology feature generation blocked for {len(engine_failures)} subjects. "
        f"See {failure_path}. Do not guess birth times or delete subjects."
    )

year_features = pd.DataFrame(rows)
feature_cols = [c for c in year_features.columns if c not in {"subject_id","year"}]
for c in feature_cols:
    year_features[c] = pd.to_numeric(year_features[c],errors="coerce").fillna(0.0)

YEAR_FEATURES = ART / "V5_DISCOVERY_ASTROLOGY_YEAR_FEATURES.csv"
year_features.to_csv(YEAR_FEATURES,index=False)

print("year-feature rows:",len(year_features))
print("astrology primitive columns:",len(feature_cols))


[SAJU_DEBUG] original_input: 1963-07-04 07:54
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1963-07-04
[SAJU_DEBUG] final_datetime(KST): 1963-07-04T07:54:00+09:00
[SAJU_DEBUG] pillars: 연=癸卯 월=戊午 일=戊申 시=丙辰
[SAJU_DEBUG] original_input: 1913-01-09 21:43
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1913-01-09
[SAJU_DEBUG] final_datetime(KST): 1913-01-09T21:43:00+09:00
[SAJU_DEBUG] pillars: 연=壬子 월=癸丑 일=庚寅 시=丁亥
[SAJU_DEBUG] original_input: 1922-03-27 23:33
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1922-03-27
[SAJU_DEBUG] final_datetime(KST): 1922-03-27T23:33:00+09:00
[SAJU_DEBUG] pillars: 연=壬戌 월=癸卯 일=乙未 시=丙子
[SAJU_DEBUG] original_input: 1921-02-24 01:17
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

engine subjects: 10 / 104


[SAJU_DEBUG] original_input: 1960-08-12 02:19
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1960-08-12
[SAJU_DEBUG] final_datetime(KST): 1960-08-12T02:19:00+09:00
[SAJU_DEBUG] pillars: 연=庚子 월=甲申 일=壬申 시=辛丑
[SAJU_DEBUG] original_input: 1949-04-20 10:19
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1949-04-20
[SAJU_DEBUG] final_datetime(KST): 1949-04-20T10:19:00+09:00
[SAJU_DEBUG] pillars: 연=己丑 월=戊辰 일=庚辰 시=辛巳
[SAJU_DEBUG] original_input: 1902-04-18 04:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1902-04-18
[SAJU_DEBUG] final_datetime(KST): 1902-04-18T04:32:00+09:00
[SAJU_DEBUG] pillars: 연=壬寅 월=甲辰 일=辛未 시=庚寅
[SAJU_DEBUG] original_input: 1900-01-08 13:00
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

engine subjects: 20 / 104


[SAJU_DEBUG] original_input: 1951-05-25 15:00
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1951-05-25
[SAJU_DEBUG] final_datetime(KST): 1951-05-25T15:00:00+09:00
[SAJU_DEBUG] 반시보정: 甲申→癸未 (15:00)
[SAJU_DEBUG] pillars: 연=辛卯 월=癸巳 일=乙丑 시=癸未
[SAJU_DEBUG] original_input: 1972-12-26 05:45
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1972-12-26
[SAJU_DEBUG] final_datetime(KST): 1972-12-26T05:45:00+09:00
[SAJU_DEBUG] pillars: 연=壬子 월=壬子 일=辛卯 시=辛卯
[SAJU_DEBUG] original_input: 1959-08-10 15:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1959-08-10
[SAJU_DEBUG] final_datetime(KST): 1959-08-10T15:32:00+09:00
[SAJU_DEBUG] pillars: 연=己亥 월=壬申 일=甲子 시=壬申
[SAJU_DEBUG] original_input: 1906-10-08 23:57
[SAJU_DEBUG] calendar=solar, is_l

...

engine subjects: 90 / 104


[SAJU_DEBUG] original_input: 1909-04-17 08:00
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1909-04-17
[SAJU_DEBUG] final_datetime(KST): 1909-04-17T08:00:00+09:00
[SAJU_DEBUG] pillars: 연=己酉 월=戊辰 일=丁未 시=甲辰
[SAJU_DEBUG] original_input: 1905-04-01 14:48
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1905-04-01
[SAJU_DEBUG] final_datetime(KST): 1905-04-01T14:48:00+09:00
[SAJU_DEBUG] pillars: 연=乙巳 월=己卯 일=庚午 시=癸未
[SAJU_DEBUG] original_input: 1940-01-16 12:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1940-01-16
[SAJU_DEBUG] final_datetime(KST): 1940-01-16T12:32:00+09:00
[SAJU_DEBUG] pillars: 연=己卯 월=丁丑 일=戊午 시=戊午
[SAJU_DEBUG] original_input: 1905-06-21 08:00
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

engine subjects: 100 / 104


[SAJU_DEBUG] original_input: 1914-10-02 20:52
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1914-10-02
[SAJU_DEBUG] final_datetime(KST): 1914-10-02T20:52:00+09:00
[SAJU_DEBUG] pillars: 연=甲寅 월=癸酉 일=辛酉 시=戊戌
[SAJU_DEBUG] original_input: 1928-09-29 17:55
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1928-09-29
[SAJU_DEBUG] final_datetime(KST): 1928-09-29T17:55:00+09:00
[SAJU_DEBUG] pillars: 연=戊辰 월=辛酉 일=壬申 시=己酉
[SAJU_DEBUG] original_input: 1930-04-09 00:12
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1930-04-09
[SAJU_DEBUG] final_datetime(KST): 1930-04-09T00:12:00+09:00
[SAJU_DEBUG] pillars: 연=庚午 월=庚辰 일=己丑 시=甲子
[SAJU_DEBUG] original_input: 1936-08-01 19:42
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

year-feature rows: 284
astrology primitive columns: 203


## 6. Pair-difference tables + nuisance representation

In [12]:

def make_pair_diff(frame, dataset_name):
    lookup = year_features.set_index(["subject_id","year"])
    out_rows = []
    axes = sorted(frame.preassigned_axis.unique())
    waves = sorted(frame.collection_wave.unique())

    for idx, row in frame.reset_index(drop=True).iterrows():
        p = lookup.loc[(str(row.subject_id), int(row.positive_year))]
        n = lookup.loc[(str(row.subject_id), int(row.negative_year))]

        out = row.to_dict()
        out["dataset"] = dataset_name
        out["pair_id"] = f"{dataset_name}_{idx+1:04d}"

        for c in feature_cols:
            out["diff__" + c] = float(p[c] - n[c])

        # Signed chronology nuisance representation.
        year_delta = float(row.positive_year - row.negative_year)
        midpoint = 0.5 * float(row.positive_year + row.negative_year)
        birth_year = float(row.birth_year)
        gap = float(row.abs_year_gap)

        out["nuisance__year_delta"] = year_delta
        out["nuisance__year_delta_x_gap"] = year_delta * gap
        out["nuisance__year_delta_x_midpoint"] = year_delta * midpoint
        out["nuisance__year_delta_x_birth_year"] = year_delta * birth_year

        for axis in axes:
            out[f"nuisance__year_delta_x_axis__{axis}"] = (
                year_delta * float(row.preassigned_axis == axis)
            )
        for wave in waves:
            safe = re.sub(r"[^A-Za-z0-9_]+","_",str(wave))
            out[f"nuisance__year_delta_x_wave__{safe}"] = (
                year_delta * float(row.collection_wave == wave)
            )

        out_rows.append(out)

    z = pd.DataFrame(out_rows)
    counts = z.groupby("subject_id").size().rename("_n")
    z = z.merge(counts,on="subject_id",how="left")
    z["subject_equal_pair_weight"] = 1.0 / z["_n"]
    return z.drop(columns="_n")

pair_primary = make_pair_diff(primary,"PRIMARY_TRUSTED")
pair_broad = make_pair_diff(broad,"BROAD_SENSITIVITY")

PAIR_PRIMARY = ART / "V5_DISCOVERY_PAIR_DIFF_PRIMARY.csv"
PAIR_BROAD = ART / "V5_DISCOVERY_PAIR_DIFF_BROAD.csv"
pair_primary.to_csv(PAIR_PRIMARY,index=False)
pair_broad.to_csv(PAIR_BROAD,index=False)

print("PRIMARY pair diff:",pair_primary.shape)
print("BROAD pair diff  :",pair_broad.shape)


PRIMARY pair diff: (193, 242)
BROAD pair diff  : (197, 242)


## 7. One frozen astrology universe + one diagnostic subset

In [13]:

ALL_DIFF = [c for c in pair_primary.columns if c.startswith("diff__")]

def pref(prefix):
    return sorted([c for c in ALL_DIFF if c.startswith("diff__"+prefix)])

TG10 = sorted([c for c in ALL_DIFF if c.startswith("diff__tg10__")])
REL = pref("relation__")
BASIC_DYNAMIC = pref("basic__")
YONG = pref("yongshin__")
UNSEONG = pref("unseong__")
TG_STRENGTH = pref("tgxstrength__")

# Primary universe: broad, deterministic, theory-bounded, fixed before scores.
ALL_ORTHODOX = sorted(set(
    TG10 + REL + BASIC_DYNAMIC + YONG + UNSEONG + TG_STRENGTH
))
NUISANCE = sorted([c for c in pair_primary.columns if c.startswith("nuisance__")])

FEATURE_SETS = {
    "TG10_RIDGE_DIAGNOSTIC": TG10,
    "ALL_ORTHODOX_RIDGE_AUTO": ALL_ORTHODOX,
    "ALL_ORTHODOX_ELASTICNET_AUTO": ALL_ORTHODOX,
}
WINNER_ELIGIBLE = [
    "ALL_ORTHODOX_RIDGE_AUTO",
    "ALL_ORTHODOX_ELASTICNET_AUTO",
]

assert ALL_ORTHODOX
assert TG10
print("TG10 diagnostic:",len(TG10))
print("ALL_ORTHODOX frozen universe:",len(ALL_ORTHODOX))
print("NUISANCE diagnostic:",len(NUISANCE))


TG10 diagnostic: 40
ALL_ORTHODOX frozen universe: 203
NUISANCE diagnostic: 10


## 8. Subject-disjoint balanced folds

In [14]:

def make_subject_folds(frame,n_folds,seed):
    rng = np.random.RandomState(int(seed))
    axes = sorted(frame.preassigned_axis.unique())
    waves = sorted(frame.collection_wave.unique())

    subject_rows = []
    for sid,g in frame.groupby("subject_id"):
        row = {
            "subject_id":str(sid),
            "n":len(g),
            "positive_earlier":int(g.positive_earlier_calc.sum()),
            "negative_earlier":int((1-g.positive_earlier_calc).sum())
        }
        for x in axes:
            row["axis__"+str(x)] = int((g.preassigned_axis==x).sum())
        for x in waves:
            row["wave__"+re.sub(r"[^A-Za-z0-9_]+","_",str(x))] = int(
                (g.collection_wave==x).sum()
            )
        subject_rows.append(row)

    meta = pd.DataFrame(subject_rows)
    n_subjects = len(meta)
    n_folds = min(int(n_folds),n_subjects)
    if n_folds < 2:
        raise ValueError("Need >=2 subjects for grouped CV.")

    meta["jitter"] = rng.uniform(size=n_subjects)
    meta = meta.sort_values(["n","jitter"],ascending=[False,True]).reset_index(drop=True)

    count_cols = [c for c in meta.columns if c not in {"subject_id","jitter"}]
    totals = meta[count_cols].sum().to_numpy(dtype=float)
    targets = totals / float(n_folds)
    scale = np.maximum(targets,1.0)

    fold_sums = np.zeros((n_folds,len(count_cols)))
    folds = [[] for _ in range(n_folds)]
    subject_target = n_subjects / float(n_folds)

    for k in range(n_folds):
        row = meta.iloc[k]
        folds[k].append(str(row.subject_id))
        fold_sums[k] += row[count_cols].to_numpy(dtype=float)

    for idx in range(n_folds,n_subjects):
        row = meta.iloc[idx]
        vec = row[count_cols].to_numpy(dtype=float)
        costs = []
        for k in range(n_folds):
            before = float(np.sum(((fold_sums[k]-targets)/scale)**2))
            after = float(np.sum(((fold_sums[k]+vec-targets)/scale)**2))
            before_n = ((len(folds[k])-subject_target)/max(subject_target,1.0))**2
            after_n = ((len(folds[k])+1-subject_target)/max(subject_target,1.0))**2
            costs.append((after-before)+0.10*(after_n-before_n)+1e-9*rng.uniform())
        best = int(np.argmin(costs))
        folds[best].append(str(row.subject_id))
        fold_sums[best] += vec

    assigned = [sid for fold in folds for sid in fold]
    assert all(folds)
    assert len(assigned)==n_subjects
    assert len(set(assigned))==n_subjects
    assert set(assigned)==set(meta.subject_id.astype(str))
    return folds

def assert_folds(frame,folds,label):
    all_s = set(frame.subject_id.astype(str))
    seen = []
    for k,fold in enumerate(folds):
        te = frame[frame.subject_id.astype(str).isin(fold)]
        tr = frame[~frame.subject_id.astype(str).isin(fold)]
        assert len(fold)>0 and not te.empty and not tr.empty
        assert set(te.subject_id.astype(str)).isdisjoint(set(tr.subject_id.astype(str)))
        seen += list(map(str,fold))
    assert set(seen)==all_s and len(seen)==len(all_s)
    return True


## 9. Pairwise model helpers

In [15]:

def train_only_feature_cleanup(train, columns):
    """Unsupervised cleanup only: zero variance + exact duplicate columns.
    Never looks at outcome labels or test fold."""
    X = train[columns].to_numpy(dtype=float)
    keep = []
    signatures = set()
    for j,c in enumerate(columns):
        col = X[:,j]
        if np.nanmax(col) - np.nanmin(col) <= 1e-12:
            continue
        sig = np.round(col,12).tobytes()
        # exact duplicate or exact negative duplicate are redundant for linear pair models
        negsig = np.round(-col,12).tobytes()
        if sig in signatures or negsig in signatures:
            continue
        signatures.add(sig)
        keep.append(c)
    if not keep:
        raise RuntimeError("All columns removed by train-only unsupervised cleanup.")
    return keep

def symmetric_xy(frame,columns):
    X = frame[columns].to_numpy(dtype=float)
    y = np.r_[np.ones(len(X),dtype=int),np.zeros(len(X),dtype=int)]
    X2 = np.vstack([X,-X])
    w0 = frame["subject_equal_pair_weight"].to_numpy(dtype=float)/2.0
    w = np.r_[w0,w0]
    return X2,y,w

def fit_l2(frame,columns,C=0.3,seed=SEED):
    X,y,w = symmetric_xy(frame,columns)
    model = Pipeline([
        ("scale",StandardScaler()),
        ("model",LogisticRegression(
            penalty="l2",C=float(C),solver="liblinear",
            max_iter=5000,random_state=int(seed)
        ))
    ])
    model.fit(X,y,model__sample_weight=w)
    return model

def fit_elastic(frame,columns,C,l1_ratio,seed):
    X,y,w = symmetric_xy(frame,columns)
    model = Pipeline([
        ("scale",StandardScaler()),
        ("model",LogisticRegression(
            penalty="elasticnet",solver="saga",C=float(C),
            l1_ratio=float(l1_ratio),max_iter=8000,random_state=int(seed)
        ))
    ])
    model.fit(X,y,model__sample_weight=w)
    return model

def decision_scores(model,frame,columns):
    return np.asarray(
        model.decision_function(frame[columns].to_numpy(dtype=float)),
        dtype=float
    )

def correct_from_score(score,eps=1e-12):
    score=np.asarray(score,dtype=float)
    return np.where(score>eps,1.0,np.where(score<-eps,0.0,0.5))

def subject_macro(frame,correct):
    z=frame[["subject_id"]].copy()
    z["correct"]=np.asarray(correct,dtype=float)
    return float(z.groupby("subject_id").correct.mean().mean())

def tune_ridge(train,columns,seed):
    cols = train_only_feature_cleanup(train,columns)
    folds = make_subject_folds(train,min(INNER_FOLDS,train.subject_id.nunique()),seed)
    assert_folds(train,folds,"INNER_RIDGE")
    rows=[]
    for C in RIDGE_C_GRID:
        vals=[]
        for k,test_s in enumerate(folds):
            tr=train[~train.subject_id.astype(str).isin(test_s)]
            te=train[train.subject_id.astype(str).isin(test_s)]
            # IMPORTANT: cleanup is redone inside each inner-training fold.
            inner_cols=train_only_feature_cleanup(tr,cols)
            model=fit_l2(tr,inner_cols,C=C,seed=seed+k)
            vals.append(subject_macro(
                te,correct_from_score(decision_scores(model,te,inner_cols))
            ))
        rows.append({
            "C":C,
            "inner_macro":float(np.mean(vals)),
            "inner_p10":float(np.quantile(vals,.10))
        })
    grid=pd.DataFrame(rows).sort_values(
        ["inner_macro","inner_p10","C"],ascending=[False,False,True]
    ).reset_index(drop=True)
    return grid.iloc[0].to_dict(),grid,cols

def tune_elastic(train,columns,seed):
    cols = train_only_feature_cleanup(train,columns)
    folds = make_subject_folds(train,min(INNER_FOLDS,train.subject_id.nunique()),seed)
    assert_folds(train,folds,"INNER_ELASTIC")
    rows=[]
    for C in ELASTIC_C_GRID:
        for l1 in ELASTIC_L1_GRID:
            vals=[]
            for k,test_s in enumerate(folds):
                tr=train[~train.subject_id.astype(str).isin(test_s)]
                te=train[train.subject_id.astype(str).isin(test_s)]
                inner_cols=train_only_feature_cleanup(tr,cols)
                model=fit_elastic(tr,inner_cols,C,l1,seed+k)
                vals.append(subject_macro(
                    te,correct_from_score(decision_scores(model,te,inner_cols))
                ))
            rows.append({
                "C":C,"l1_ratio":l1,
                "inner_macro":float(np.mean(vals)),
                "inner_p10":float(np.quantile(vals,.10))
            })
    grid=pd.DataFrame(rows).sort_values(
        ["inner_macro","inner_p10","C"],ascending=[False,False,True]
    ).reset_index(drop=True)
    return grid.iloc[0].to_dict(),grid,cols


## 10. Repeated outer CV on PRIMARY

In [16]:

CANDIDATES = list(FEATURE_SETS)
pair_oof=[]
subject_oof=[]
fold_rows=[]
inner_grid_rows=[]
elastic_coef_rows=[]

for repeat in range(OUTER_REPEATS):
    folds=make_subject_folds(pair_primary,OUTER_FOLDS,SEED+1000*repeat)
    assert_folds(pair_primary,folds,f"OUTER_{repeat}")

    for fold_i,test_s in enumerate(folds):
        train=pair_primary[~pair_primary.subject_id.astype(str).isin(test_s)].copy()
        test=pair_primary[pair_primary.subject_id.astype(str).isin(test_s)].copy()

        diag={
            "repeat":repeat,"fold":fold_i,
            "n_test_subjects":int(test.subject_id.nunique()),
            "n_test_pairs":int(len(test)),
            "positive_earlier_share":float(test.positive_earlier_calc.mean())
        }
        for axis in sorted(pair_primary.preassigned_axis.unique()):
            diag["axis__"+axis]=int((test.preassigned_axis==axis).sum())
        for wave in sorted(pair_primary.collection_wave.unique()):
            diag["wave__"+str(wave)]=int((test.collection_wave==wave).sum())
        fold_rows.append(diag)

        # Nuisance is diagnostic only.
        nuisance_cols=train_only_feature_cleanup(train,NUISANCE)
        best_n,grid_n,_=tune_ridge(
            train,nuisance_cols,SEED+500000+1000*repeat+fold_i
        )
        nuisance_model=fit_l2(
            train,nuisance_cols,C=best_n["C"],seed=SEED+repeat*100+fold_i
        )
        scored={
            "NUISANCE_ONLY":correct_from_score(
                decision_scores(nuisance_model,test,nuisance_cols)
            )
        }

        for name,astro_universe in FEATURE_SETS.items():
            # TG10 diagnostic is fixed Ridge-auto. Both ALL models are winner-eligible.
            if name in ["TG10_RIDGE_DIAGNOSTIC","ALL_ORTHODOX_RIDGE_AUTO"]:
                best,grid,astro_cols=tune_ridge(
                    train,astro_universe,SEED+10000*repeat+fold_i
                )
                grid["repeat"]=repeat; grid["fold"]=fold_i
                grid["model"]=name; grid["kind"]="ASTRO"
                inner_grid_rows.append(grid)

                a_model=fit_l2(
                    train,astro_cols,C=best["C"],seed=SEED+20000*repeat+fold_i
                )

                na_universe=sorted(set(NUISANCE+astro_universe))
                best_na,grid_na,na_cols=tune_ridge(
                    train,na_universe,SEED+30000*repeat+fold_i
                )
                grid_na["repeat"]=repeat; grid_na["fold"]=fold_i
                grid_na["model"]=name; grid_na["kind"]="NPLUS"
                inner_grid_rows.append(grid_na)

                na_model=fit_l2(
                    train,na_cols,C=best_na["C"],seed=SEED+40000*repeat+fold_i
                )

            elif name=="ALL_ORTHODOX_ELASTICNET_AUTO":
                best,grid,astro_cols=tune_elastic(
                    train,astro_universe,SEED+10000*repeat+fold_i
                )
                grid["repeat"]=repeat; grid["fold"]=fold_i
                grid["model"]=name; grid["kind"]="ASTRO"
                inner_grid_rows.append(grid)

                a_model=fit_elastic(
                    train,astro_cols,best["C"],best["l1_ratio"],
                    SEED+20000*repeat+fold_i
                )

                na_universe=sorted(set(NUISANCE+astro_universe))
                best_na,grid_na,na_cols=tune_elastic(
                    train,na_universe,SEED+30000*repeat+fold_i
                )
                grid_na["repeat"]=repeat; grid_na["fold"]=fold_i
                grid_na["model"]=name; grid_na["kind"]="NPLUS"
                inner_grid_rows.append(grid_na)

                na_model=fit_elastic(
                    train,na_cols,best_na["C"],best_na["l1_ratio"],
                    SEED+40000*repeat+fold_i
                )

                coef=a_model.named_steps["model"].coef_.ravel()
                for col,val in zip(astro_cols,coef):
                    elastic_coef_rows.append({
                        "repeat":repeat,"fold":fold_i,
                        "feature":col,"coefficient":float(val),
                        "selected":float(abs(val)>1e-8)
                    })
            else:
                raise RuntimeError(name)

            scored["ASTRO__"+name]=correct_from_score(
                decision_scores(a_model,test,astro_cols)
            )
            scored["NPLUS__"+name]=correct_from_score(
                decision_scores(na_model,test,na_cols)
            )

        for model_name,correct in scored.items():
            tmp=test[[
                "pair_id","subject_id","preassigned_axis",
                "collection_wave","positive_earlier_calc"
            ]].copy()
            tmp["repeat"]=repeat
            tmp["fold"]=fold_i
            tmp["model"]=model_name
            tmp["correct"]=correct
            pair_oof.append(tmp)

            sm=tmp.groupby("subject_id").correct.mean().reset_index()
            sm["repeat"]=repeat
            sm["fold"]=fold_i
            sm["model"]=model_name
            subject_oof.append(sm)

pair_oof=pd.concat(pair_oof,ignore_index=True)
subject_oof=pd.concat(subject_oof,ignore_index=True)
fold_df=pd.DataFrame(fold_rows)

pair_oof.to_csv(ART/"V5_DISCOVERY_PRIMARY_OOF_PAIR_SCORES.csv",index=False)
subject_oof.to_csv(ART/"V5_DISCOVERY_PRIMARY_OOF_SUBJECT_SCORES.csv",index=False)
fold_df.to_csv(ART/"V5_DISCOVERY_OUTER_FOLD_BALANCE.csv",index=False)

inner_grid=pd.concat(inner_grid_rows,ignore_index=True) if inner_grid_rows else pd.DataFrame()
elastic_coef=pd.DataFrame(elastic_coef_rows)
inner_grid.to_csv(ART/"V5_DISCOVERY_AUTOSELECT_INNER_GRID.csv",index=False)
elastic_coef.to_csv(ART/"V5_DISCOVERY_ELASTICNET_FOLD_COEFFICIENTS.csv",index=False)

if len(elastic_coef):
    selection_frequency=(
        elastic_coef.groupby("feature")
        .agg(
            selection_frequency=("selected","mean"),
            mean_coefficient=("coefficient","mean"),
            mean_abs_coefficient=("coefficient",lambda s: float(np.mean(np.abs(s))))
        )
        .reset_index()
        .sort_values(["selection_frequency","mean_abs_coefficient"],ascending=[False,False])
    )
else:
    selection_frequency=pd.DataFrame(
        columns=["feature","selection_frequency","mean_coefficient","mean_abs_coefficient"]
    )
selection_frequency.to_csv(
    ART/"V5_DISCOVERY_ELASTICNET_SELECTION_STABILITY.csv",index=False
)

print("PRIMARY automatic-selection tournament complete")
display(fold_df)
display(selection_frequency.head(40))


PRIMARY automatic-selection tournament complete


,repeat,fold,n_test_subjects,n_test_pairs,positive_earlier_share,axis__COMPETITIVE,axis__PROJECT,axis__STATUS,wave__E1,wave__E2,wave__ORIGINAL
0,0,0,19,38,0.631579,27,2,9,12,4,22
1,0,1,19,38,0.631579,27,2,9,11,4,23
2,0,2,20,39,0.615385,27,3,9,12,5,22
3,0,3,21,39,0.615385,26,3,10,11,5,23
4,0,4,21,39,0.615385,27,2,10,11,5,23
5,1,0,20,39,0.615385,28,2,9,11,5,23
6,1,1,19,38,0.631579,27,2,9,11,5,22
7,1,2,21,39,0.615385,26,3,10,12,4,23
8,1,3,20,38,0.631579,26,2,10,11,4,23
9,1,4,20,39,0.615385,27,3,9,12,5,22


,feature,selection_frequency,mean_coefficient,mean_abs_coefficient
71,diff__tg10__dw_stem_pian_cai,0.76,0.343695,0.343695
30,diff__basic__sw_branch_酉,0.72,-0.344772,0.344772
69,diff__tg10__dw_stem_bijian,0.72,0.172212,0.172212
155,diff__unseong__sw_state_건록,0.68,-0.167938,0.167938
51,diff__relation__sw_dw_branch_clash,0.64,0.336592,0.336592
122,diff__tgxstrength__sw_jiecai_weak,0.64,0.226503,0.226503
129,diff__tgxstrength__sw_shangguan_strong,0.64,0.216321,0.216321
25,diff__basic__sw_branch_巳,0.64,0.207745,0.207745
128,diff__tgxstrength__sw_qi_sha_weak,0.64,-0.154046,0.154046
113,diff__tgxstrength__dw_zheng_cai_strong,0.64,-0.090687,0.090687


## 11. PRIMARY leaderboard + conditional nuisance lift

In [17]:

repeat_metrics=(
    subject_oof.groupby(["model","repeat"]).correct.mean()
    .reset_index(name="subject_macro")
)
leader_rows=[]
for model,g in repeat_metrics.groupby("model"):
    vals=g.subject_macro.to_numpy(dtype=float)
    leader_rows.append({
        "model":model,
        "subject_macro_mean":float(vals.mean()),
        "subject_macro_std":float(vals.std(ddof=1)) if len(vals)>1 else 0.0,
        "repeat_p10":float(np.quantile(vals,.10)),
        "repeat_min":float(vals.min())
    })
leader=pd.DataFrame(leader_rows).sort_values("subject_macro_mean",ascending=False)
leader.to_csv(ART/"V5_DISCOVERY_PRIMARY_LEADERBOARD.csv",index=False)
display(leader)

# Subject-level repeated OOF average for paired bootstrap.
subject_avg=(
    subject_oof.groupby(["model","subject_id"]).correct.mean().reset_index()
)
wide=subject_avg.pivot(index="subject_id",columns="model",values="correct")
rng=np.random.RandomState(SEED+999)

boot_rows=[]
for candidate in CANDIDATES:
    nplus="NPLUS__"+candidate
    if nplus not in wide.columns or "NUISANCE_ONLY" not in wide.columns:
        continue
    shared=wide[[nplus,"NUISANCE_ONLY"]].dropna()
    ids=shared.index.to_numpy()
    observed=float((shared[nplus]-shared["NUISANCE_ONLY"]).mean())
    vals=[]
    for _ in range(N_BOOTSTRAP):
        sample=rng.choice(ids,size=len(ids),replace=True)
        vals.append(float(
            (shared.loc[sample,nplus].to_numpy()
             -shared.loc[sample,"NUISANCE_ONLY"].to_numpy()).mean()
        ))
    arr=np.asarray(vals)
    boot_rows.append({
        "candidate":candidate,
        "reference":"NUISANCE_ONLY",
        "observed_delta":observed,
        "bootstrap_mean_delta":float(arr.mean()),
        "ci025":float(np.quantile(arr,.025)),
        "ci975":float(np.quantile(arr,.975)),
        "p_delta_gt_0":float((arr>0).mean())
    })
boot=pd.DataFrame(boot_rows)
boot.to_csv(ART/"V5_DISCOVERY_NUISANCE_INCREMENT_BOOTSTRAP.csv",index=False)
display(boot)


,model,subject_macro_mean,subject_macro_std,repeat_p10,repeat_min
6,NUISANCE_ONLY,0.677733,0.023669,0.657845,0.655845
5,NPLUS__TG10_RIDGE_DIAGNOSTIC,0.655764,0.023779,0.633686,0.618119
3,NPLUS__ALL_ORTHODOX_ELASTICNET_AUTO,0.624945,0.093453,0.525052,0.470833
4,NPLUS__ALL_ORTHODOX_RIDGE_AUTO,0.618805,0.043413,0.584719,0.584452
2,ASTRO__TG10_RIDGE_DIAGNOSTIC,0.544274,0.008875,0.534912,0.532274
1,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,0.536943,0.020508,0.515762,0.503786
0,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,0.464443,0.076828,0.389983,0.340417


,candidate,reference,observed_delta,bootstrap_mean_delta,ci025,ci975,p_delta_gt_0
0,TG10_RIDGE_DIAGNOSTIC,NUISANCE_ONLY,-0.021969,-0.021935,-0.105671,0.061290,0.3047
1,ALL_ORTHODOX_RIDGE_AUTO,NUISANCE_ONLY,-0.058929,-0.059183,-0.144364,0.025147,0.0818
2,ALL_ORTHODOX_ELASTICNET_AUTO,NUISANCE_ONLY,-0.052788,-0.052676,-0.117739,0.014000,0.0614


## 12. PRIMARY wave and axis robustness

In [18]:

pair_avg=(
    pair_oof.groupby([
        "model","pair_id","subject_id","preassigned_axis","collection_wave"
    ]).correct.mean().reset_index()
)

wave_rows=[]
for (model,wave),g in pair_avg.groupby(["model","collection_wave"]):
    subj=g.groupby("subject_id").correct.mean()
    wave_rows.append({
        "model":model,"collection_wave":wave,
        "n_pairs":len(g),"n_subjects":g.subject_id.nunique(),
        "subject_macro":float(subj.mean())
    })
wave_df=pd.DataFrame(wave_rows)
wave_df.to_csv(ART/"V5_DISCOVERY_WAVE_ROBUSTNESS.csv",index=False)

axis_rows=[]
for (model,axis),g in pair_avg.groupby(["model","preassigned_axis"]):
    subj=g.groupby("subject_id").correct.mean()
    axis_rows.append({
        "model":model,"axis":axis,
        "n_pairs":len(g),"n_subjects":g.subject_id.nunique(),
        "subject_macro":float(subj.mean())
    })
axis_df=pd.DataFrame(axis_rows)
axis_df.to_csv(ART/"V5_DISCOVERY_AXIS_DIAGNOSTICS.csv",index=False)

display(wave_df)
display(axis_df)


,model,collection_wave,n_pairs,n_subjects,subject_macro
0,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,E1,57,29,0.452299
1,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,E2,23,18,0.444444
2,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,ORIGINAL,113,53,0.477880
3,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,E1,57,29,0.547126
4,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,E2,23,18,0.472222
5,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,ORIGINAL,113,53,0.553351
6,ASTRO__TG10_RIDGE_DIAGNOSTIC,E1,57,29,0.540805
7,ASTRO__TG10_RIDGE_DIAGNOSTIC,E2,23,18,0.555556
8,ASTRO__TG10_RIDGE_DIAGNOSTIC,ORIGINAL,113,53,0.542341
9,NPLUS__ALL_ORTHODOX_ELASTICNET_AUTO,E1,57,29,0.597126


,model,axis,n_pairs,n_subjects,subject_macro
0,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,COMPETITIVE,134,57,0.450777
1,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,PROJECT,12,9,0.396296
2,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,STATUS,47,34,0.505392
3,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,COMPETITIVE,134,57,0.487911
4,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,PROJECT,12,9,0.559259
5,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,STATUS,47,34,0.613235
6,ASTRO__TG10_RIDGE_DIAGNOSTIC,COMPETITIVE,134,57,0.533521
7,ASTRO__TG10_RIDGE_DIAGNOSTIC,PROJECT,12,9,0.614815
8,ASTRO__TG10_RIDGE_DIAGNOSTIC,STATUS,47,34,0.543627
9,NPLUS__ALL_ORTHODOX_ELASTICNET_AUTO,COMPETITIVE,134,57,0.499612



## 13. BROAD sensitivity

BROAD is **not** used to invent a new architecture.  
The exact same predefined modeling algorithm is applied only as a robustness sensitivity.


In [19]:

def simple_repeated_cv(frame,feature_sets,repeats=5):
    subject_rows=[]

    for repeat in range(repeats):
        folds=make_subject_folds(frame,OUTER_FOLDS,SEED+50000+1000*repeat)
        assert_folds(frame,folds,f"BROAD_{repeat}")

        for fold_i,test_s in enumerate(folds):
            tr=frame[~frame.subject_id.astype(str).isin(test_s)]
            te=frame[frame.subject_id.astype(str).isin(test_s)]

            best_n,_,ncols=tune_ridge(
                tr,NUISANCE,SEED+50500+1000*repeat+fold_i
            )
            nuisance_model=fit_l2(tr,ncols,C=best_n["C"])
            scored={
                "NUISANCE_ONLY":correct_from_score(
                    decision_scores(nuisance_model,te,ncols)
                )
            }

            for name,astro_universe in feature_sets.items():
                if name in ["TG10_RIDGE_DIAGNOSTIC","ALL_ORTHODOX_RIDGE_AUTO"]:
                    best,_,astro_cols=tune_ridge(
                        tr,astro_universe,SEED+60000+1000*repeat+fold_i
                    )
                    a=fit_l2(tr,astro_cols,C=best["C"])
                    best_na,_,na_cols=tune_ridge(
                        tr,sorted(set(NUISANCE+astro_universe)),
                        SEED+61000+1000*repeat+fold_i
                    )
                    na=fit_l2(tr,na_cols,C=best_na["C"])
                else:
                    best,_,astro_cols=tune_elastic(
                        tr,astro_universe,SEED+62000+1000*repeat+fold_i
                    )
                    a=fit_elastic(
                        tr,astro_cols,best["C"],best["l1_ratio"],SEED+63000+fold_i
                    )
                    best_na,_,na_cols=tune_elastic(
                        tr,sorted(set(NUISANCE+astro_universe)),
                        SEED+64000+1000*repeat+fold_i
                    )
                    na=fit_elastic(
                        tr,na_cols,best_na["C"],best_na["l1_ratio"],SEED+65000+fold_i
                    )

                scored["ASTRO__"+name]=correct_from_score(
                    decision_scores(a,te,astro_cols)
                )
                scored["NPLUS__"+name]=correct_from_score(
                    decision_scores(na,te,na_cols)
                )

            for model,correct in scored.items():
                tmp=te[["subject_id"]].copy()
                tmp["correct"]=correct
                sm=tmp.groupby("subject_id").correct.mean().reset_index()
                sm["repeat"]=repeat
                sm["model"]=model
                subject_rows.append(sm)

    sub=pd.concat(subject_rows,ignore_index=True)
    rep=sub.groupby(["model","repeat"]).correct.mean().reset_index(name="subject_macro")
    summary=(
        rep.groupby("model").subject_macro
        .agg(["mean","std","min"])
        .reset_index()
        .rename(columns={"mean":"subject_macro_mean","std":"subject_macro_std","min":"repeat_min"})
    )
    return sub,summary

broad_oof,broad_leader=simple_repeated_cv(pair_broad,FEATURE_SETS,OUTER_REPEATS)
broad_oof.to_csv(ART/"V5_DISCOVERY_BROAD_OOF_SUBJECT_SCORES.csv",index=False)
broad_leader.to_csv(ART/"V5_DISCOVERY_BROAD_LEADERBOARD.csv",index=False)
display(broad_leader)


,model,subject_macro_mean,subject_macro_std,repeat_min
0,ASTRO__ALL_ORTHODOX_ELASTICNET_AUTO,0.468471,0.181930,0.350160
1,ASTRO__ALL_ORTHODOX_RIDGE_AUTO,0.524366,0.025034,0.486676
2,ASTRO__TG10_RIDGE_DIAGNOSTIC,0.521474,0.038886,0.493693
3,NPLUS__ALL_ORTHODOX_ELASTICNET_AUTO,0.588031,0.071516,0.510897
4,NPLUS__ALL_ORTHODOX_RIDGE_AUTO,0.619673,0.036953,0.581742
5,NPLUS__TG10_RIDGE_DIAGNOSTIC,0.660893,0.036740,0.603411
6,NUISANCE_ONLY,0.707260,0.022678,0.675355


## 14. Candidate gates and architecture freeze decision

In [20]:

lead=leader.set_index("model")
broad_lookup=broad_leader.set_index("model")
boot_lookup=boot.set_index("candidate")

gate_rows=[]
SIMPLICITY_ORDER={"ALL_ORTHODOX_RIDGE_AUTO":0,"ALL_ORTHODOX_ELASTICNET_AUTO":1}

for candidate in WINNER_ELIGIBLE:
    astro="ASTRO__"+candidate
    nplus="NPLUS__"+candidate

    astro_mean=float(lead.loc[astro,"subject_macro_mean"])
    astro_p10=float(lead.loc[astro,"repeat_p10"])
    nplus_mean=float(lead.loc[nplus,"subject_macro_mean"])
    nuisance_mean=float(lead.loc["NUISANCE_ONLY","subject_macro_mean"])

    conditional_delta=float(
        boot_lookup.loc[candidate,"observed_delta"]
    )
    boot_p=float(
        boot_lookup.loc[candidate,"p_delta_gt_0"]
    )

    broad_astro=float(broad_lookup.loc[astro,"subject_macro_mean"])
    broad_nplus=float(broad_lookup.loc[nplus,"subject_macro_mean"])
    broad_nuis=float(broad_lookup.loc["NUISANCE_ONLY","subject_macro_mean"])
    broad_delta=broad_nplus-broad_nuis

    # Wave floor only for waves with >=10 subjects for this astrology model.
    wg=wave_df[wave_df.model==astro].copy()
    supported=wg[wg.n_subjects>=SUPPORTED_WAVE_MIN_SUBJECTS]
    supported_wave_min=(
        float(supported.subject_macro.min()) if len(supported) else np.nan
    )
    wave_pass=(
        True if len(supported)==0
        else supported_wave_min>=MIN_SUPPORTED_WAVE
    )

    passes={
        "astro_ge_055":astro_mean>=MIN_ASTRO_MACRO,
        "astro_p10_ge_048":astro_p10>=MIN_ASTRO_P10,
        "conditional_delta_ge_001":conditional_delta>=MIN_CONDITIONAL_DELTA,
        "bootstrap_p_delta_ge_080":boot_p>=MIN_BOOT_P_DELTA,
        "broad_astro_ge_052":broad_astro>=MIN_BROAD_ASTRO,
        "broad_conditional_delta_ge_000":broad_delta>=MIN_BROAD_DELTA,
        "supported_wave_floor_ge_045":bool(wave_pass),
    }

    gate_rows.append({
        "candidate":candidate,
        "astro_primary":astro_mean,
        "astro_primary_p10":astro_p10,
        "nplus_primary":nplus_mean,
        "nuisance_primary":nuisance_mean,
        "conditional_delta_vs_nuisance":conditional_delta,
        "bootstrap_p_delta_gt_0":boot_p,
        "broad_astro":broad_astro,
        "broad_nplus":broad_nplus,
        "broad_nuisance":broad_nuis,
        "broad_conditional_delta":broad_delta,
        "supported_wave_min":supported_wave_min,
        "simplicity_order":SIMPLICITY_ORDER[candidate],
        **passes,
        "all_gates":all(passes.values())
    })

gates=pd.DataFrame(gate_rows).sort_values(
    ["all_gates","astro_primary","conditional_delta_vs_nuisance","simplicity_order"],
    ascending=[False,False,False,True]
).reset_index(drop=True)
gates.to_csv(ART/"V5_DISCOVERY_CANDIDATE_GATES.csv",index=False)
display(gates)

survivors=gates[gates.all_gates].copy()
if len(survivors):
    winner=str(survivors.iloc[0].candidate)
    status="V5_DISCOVERY_CANDIDATE_ARCHITECTURE_FROZEN_READY_FOR_CONTROL_BENCHMARK"
    reason="At least one shared-coefficient astrology architecture passed primary, nuisance-increment, broad, and wave robustness gates."
else:
    winner=None
    status="V5_NO_ASTROLOGY_ARCHITECTURE_SURVIVES_DISCOVERY_GATE"
    reason="No shared-coefficient astrology architecture passed all frozen V5 discovery gates."

print(status,winner)


,candidate,astro_primary,astro_primary_p10,nplus_primary,nuisance_primary,conditional_delta_vs_nuisance,bootstrap_p_delta_gt_0,broad_astro,broad_nplus,broad_nuisance,...,supported_wave_min,simplicity_order,astro_ge_055,astro_p10_ge_048,conditional_delta_ge_001,bootstrap_p_delta_ge_080,broad_astro_ge_052,broad_conditional_delta_ge_000,supported_wave_floor_ge_045,all_gates
0,ALL_ORTHODOX_RIDGE_AUTO,0.536943,0.515762,0.618805,0.677733,-0.058929,0.0818,0.524366,0.619673,0.70726,...,0.472222,0,False,True,False,False,True,False,True,False
1,ALL_ORTHODOX_ELASTICNET_AUTO,0.464443,0.389983,0.624945,0.677733,-0.052788,0.0614,0.468471,0.588031,0.70726,...,0.444444,1,False,False,False,False,False,False,False,False


V5_NO_ASTROLOGY_ARCHITECTURE_SURVIVES_DISCOVERY_GATE None


## 15. If a winner survives, freeze full-PRIMARY astrology-only coefficients

In [21]:

candidate_spec_path=ART/"V5_DISCOVERY_FROZEN_CANDIDATE_MODEL_SPEC.json"
candidate_coef_path=ART/"V5_DISCOVERY_FROZEN_CANDIDATE_COEFFICIENTS.csv"

final_model_info=None
if winner is not None:
    universe=FEATURE_SETS[winner]

    if winner=="ALL_ORTHODOX_ELASTICNET_AUTO":
        best_full,full_grid,cols=tune_elastic(
            pair_primary,universe,SEED+90000
        )
        model=fit_elastic(
            pair_primary,cols,best_full["C"],best_full["l1_ratio"],SEED+91000
        )
        hyper={
            "penalty":"elasticnet",
            "C":float(best_full["C"]),
            "l1_ratio":float(best_full["l1_ratio"]),
            "hyperparameter_selection":"inner subject-CV using frozen grid"
        }
    elif winner=="ALL_ORTHODOX_RIDGE_AUTO":
        best_full,full_grid,cols=tune_ridge(
            pair_primary,universe,SEED+90000
        )
        model=fit_l2(pair_primary,cols,C=best_full["C"],seed=SEED+91000)
        hyper={
            "penalty":"l2",
            "C":float(best_full["C"]),
            "hyperparameter_selection":"inner subject-CV using frozen grid"
        }
    else:
        raise RuntimeError("Diagnostic model cannot be frozen: "+str(winner))

    scaler=model.named_steps["scale"]
    clf=model.named_steps["model"]

    coef=pd.DataFrame({
        "feature":cols,
        "coefficient":clf.coef_.ravel(),
        "scaler_mean":scaler.mean_,
        "scaler_scale":scaler.scale_
    })
    coef.to_csv(candidate_coef_path,index=False)

    spec={
        "version":"V5_DISCOVERY_FROZEN_CANDIDATE_MODEL_SPEC_V2_AUTOSELECT",
        "created_at":datetime.now().isoformat(timespec="seconds"),
        "status":"FROZEN_AFTER_DISCOVERY_BEFORE_CONTROL_AND_CONFIRM",
        "architecture":winner,
        "feature_universe":"ALL_ORTHODOX_DYNAMIC_V1",
        "universe_feature_count":len(ALL_ORTHODOX),
        "fitted_feature_count_after_unsupervised_cleanup":len(cols),
        "fitted_feature_columns":cols,
        "hyperparameters":hyper,
        "training_dataset":"PRIMARY_TRUSTED",
        "training_pairs_n":int(len(pair_primary)),
        "training_subjects_n":int(pair_primary.subject_id.nunique()),
        "manual_post_score_feature_selection":False,
        "shared_coefficients_across_axes":True,
        "uses_nuisance_features_in_production_candidate":False,
        "uses_collection_wave":False,
        "uses_control_score":False,
        "confirm_loaded":False,
        "pair_primary_sha256":sha256_file(PRIMARY_PAIRS),
        "year_features_sha256":sha256_file(YEAR_FEATURES),
        "coefficients_sha256":sha256_file(candidate_coef_path),
        "next_rule":"Benchmark frozen candidate against Production Control on DEV without retuning. Only then decide whether to open/research sealed CONFIRM."
    }
    json.dump(spec,open(candidate_spec_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)
    final_model_info=spec
else:
    if candidate_spec_path.exists():
        candidate_spec_path.unlink()
    if candidate_coef_path.exists():
        candidate_coef_path.unlink()

print("Frozen candidate written:",winner is not None)


Frozen candidate written: False


## 16. Final decision

In [22]:

payload={
    "version":"V5_DISCOVERY_ASTROLOGY_TOURNAMENT_DECISION_V2_AUTOSELECT",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":status,
    "winner":winner,
    "reason":reason,
    "target":{
        "primary_pairs_n":int(len(pair_primary)),
        "primary_subjects_n":int(pair_primary.subject_id.nunique()),
        "broad_pairs_n":int(len(pair_broad)),
        "broad_subjects_n":int(pair_broad.subject_id.nunique()),
        "readiness_decision_sha256":sha256_file(READY),
        "primary_pairs_sha256":sha256_file(PRIMARY_PAIRS),
        "broad_pairs_sha256":sha256_file(BROAD_PAIRS),
        "preregistration_sha256":sha256_file(PREREG)
    },
    "gates":gates.to_dict(orient="records"),
    "rules":{
        "event_or_pair_membership_changed":False,
        "axis_specific_astrology_coefficients_used":False,
        "collection_wave_used_as_astrology_feature":False,
        "production_control_scored":False,
        "confirm_loaded_or_researched":False,
        "exact_birth_time_required":True
    },
    "frozen_candidate_model_spec_sha256":(
        sha256_file(candidate_spec_path) if candidate_spec_path.exists() else None
    ),
    "frozen_candidate_coefficients_sha256":(
        sha256_file(candidate_coef_path) if candidate_coef_path.exists() else None
    ),
    "next_rule":(
        "If winner exists: benchmark the frozen astrology-only candidate against current Production Control on DEV with zero retuning; "
        "if it remains credible, then collect/score sealed CONFIRM once. "
        "If no winner: keep CONFIRM sealed and diagnose astrology representation on DEV only."
    )
}
decision_path=ART/"V5_DISCOVERY_ASTROLOGY_TOURNAMENT_DECISION.json"
json.dump(payload,open(decision_path,"w",encoding="utf-8"),ensure_ascii=False,indent=2)

print(json.dumps({
    "status":status,
    "winner":winner,
    "production_control_scored":False,
    "confirm_loaded":False,
    "next_rule":payload["next_rule"]
},ensure_ascii=False,indent=2))


{
  "status": "V5_NO_ASTROLOGY_ARCHITECTURE_SURVIVES_DISCOVERY_GATE",
  "winner": null,
  "production_control_scored": false,
  "confirm_loaded": false,
  "next_rule": "If winner exists: benchmark the frozen astrology-only candidate against current Production Control on DEV with zero retuning; if it remains credible, then collect/score sealed CONFIRM once. If no winner: keep CONFIRM sealed and diagnose astrology representation on DEV only."
}



## Send back after `Kernel Restart → Run All`

Send these **6 files**:

```text
V5_DISCOVERY_ASTROLOGY_TOURNAMENT_DECISION.json
V5_DISCOVERY_PRIMARY_LEADERBOARD.csv
V5_DISCOVERY_CANDIDATE_GATES.csv
V5_DISCOVERY_NUISANCE_INCREMENT_BOOTSTRAP.csv
V5_DISCOVERY_WAVE_ROBUSTNESS.csv
V5_DISCOVERY_ELASTICNET_SELECTION_STABILITY.csv
```

If a candidate freezes, also send:

```text
V5_DISCOVERY_FROZEN_CANDIDATE_MODEL_SPEC.json
V5_DISCOVERY_FROZEN_CANDIDATE_COEFFICIENTS.csv
```

Do **not** run the old 24A first.  
Do **not** open CONFIRM.
